In [ ]:
from setup import *

In [ ]:
entities = pd.read_csv("./data/metadata/all_entities.csv")

# Search Terms definieren

Über die Keyword-Suche können wir alle Treffer zu bestimmten Suchbegriffen finden. Für die Erstellung der Suchbegriff-Liste bietet es sich an, zunächst über das Dashboard Inhalte zu explorieren, und/oder Clara zu fragen.



In [ ]:
entity_ids = entities["id"].to_list()
search_terms_wp = ["Wärmeplanung", "Wärmeplan", "Fernwärme", "Fernwärmenetz", "Wärmenetz", "Wärmeversorgung", "Wärmeversorgungskonzept", "Wärmeversorgungskonzepte", "Wärmeplanungsgesetz"]
search_terms_autofrei = ["autofrei", "autofreie Stadt", "autofreies Stadtzentrum", "autofreie Innenstadt", "autofreie Innenstadtbereiche", "autofreie Innenstadtbereiche", "autofreie Innenstadtbereiche", "autofreie Innenstadtbereiche"]
search_terms_autoarm = ["autoarm", "autoarme Stadt", "autoarmes Stadtzentrum", "autoarme Innenstadt", "autoarme Innenstadtbereiche", "autoarme Innenstadtbereiche", "autoarme Innenstadtbereiche", "autoarme Innenstadtbereiche"]
search_terms = search_terms_autoarm
search_string = " OR ".join(search_terms)

In [ ]:
len(set(entity_ids))

In [ ]:
search_string

In [ ]:
# Alle Items sammeln
all_items = []

# Paginierungsparameter
limit = 499
request_count = 0  # Zähler für Anfragen
max_requests_per_minute = 29  # Nach 29 Anfragen warten

print(f"\n🔍 Durchsuche {len(entity_ids)} Entity IDs...")

offset = 0
total_items = None

print(poliscope_headers)

while True:
    response = requests.get(
        url=f"{poliscope_api_url}/search/scan",
        headers=poliscope_headers,
        params={
            "q": search_string,
            "limit": limit,
            "offset": offset,
            "entityIds": ["03*"] # Ganz Niedersachsen
        }
    )

    request_count += 1

    if response.status_code != 200:
        print(f"  ✗ Error: {response.status_code}")
        print(f"  ✗ Error: {response.json()}")
        break

    data = response.json()
    items = data.get("data", [])

    meta = data.get("meta", {})
    pagination = meta.get("pagination", {})
    total_items = pagination.get("total", 0)

    if not items:
        print("  → Keine weiteren Treffer.")
        break

    all_items.extend(items)
    offset += len(items)
    print(f"  → Downloaded {offset} / {total_items} items (Request #{request_count})")

    if offset >= total_items or len(items) < limit:
        break

    if request_count >= max_requests_per_minute:
        print("  ⏳ Rate limit erreicht. Warte 1 Minute...")
        time.sleep(60)
        request_count = 0

print(f"\n✓ FERTIG! Insgesamt {len(all_items)} items abgerufen.")

In [ ]:
all_items_df = pd.DataFrame(all_items)
all_items_df


In [ ]:
all_items_df.to_csv("./data/raw/autoarm_items.csv", index=False)

highlights: Sind die Character Positions in dem text string

agendaItemID ist die Verknüpfung zu einer Sitzung

vmtl ist proposalID leer, wenns keins gibt